In [1]:
!pip install transformers datasets accelerate scikit-learn -q

In [2]:
import ast
import csv
import random
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    accuracy_score,
)

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from datasets import Dataset

In [3]:
def load_data(data_path):

    labeled_data = []

    with open(data_path, "r", encoding="utf-8") as f:

        reader = csv.DictReader(f)

        for row in reader:

            overview = row["overview"]
            genre_string = row["genre"]

            if overview is None or genre_string is None:
                continue

            overview = overview.strip()
            genre_string = genre_string.strip()

            if overview == "" or genre_string == "":
                continue

            try:
                genre_list = ast.literal_eval(
                    genre_string
                )
            except:
                continue

            if len(genre_list) == 0:
                continue

            labeled_data.append(
                (overview, genre_list)
            )

    return labeled_data


def shuffle_data(
    labeled_data,
    seed=42,
):
    random.seed(seed)
    random.shuffle(labeled_data)
    return labeled_data


def split_data(
    texts,
    labels,
    train_ratio=0.6,
    val_ratio=0.3,
    test_ratio=0.1,
):

    train_count = int(
        len(texts) * train_ratio
    )

    val_count = int(
        len(texts) * val_ratio
    )

    train_texts = texts[:train_count]

    val_texts = texts[
        train_count:
        train_count + val_count
    ]

    test_texts = texts[
        train_count + val_count:
    ]

    train_labels = labels[:train_count]

    val_labels = labels[
        train_count:
        train_count + val_count
    ]

    test_labels = labels[
        train_count + val_count:
    ]

    return (
        train_texts,
        val_texts,
        test_texts,
        train_labels,
        val_labels,
        test_labels,
    )


def load_data_for_bert(data_path):

    labeled_data = load_data(
        data_path
    )

    labeled_data = shuffle_data(
        labeled_data
    )

    texts = [
        x[0]
        for x in labeled_data
    ]

    labels = [
        x[1]
        for x in labeled_data
    ]

    (
        train_texts,
        val_texts,
        test_texts,
        train_labels,
        val_labels,
        test_labels,
    ) = split_data(
        texts,
        labels
    )

    mlb = MultiLabelBinarizer()

    y_train = mlb.fit_transform(
        train_labels
    )

    y_val = mlb.transform(
        val_labels
    )

    y_test = mlb.transform(
        test_labels
    )

    return (
        train_texts,
        val_texts,
        test_texts,
        y_train,
        y_val,
        y_test,
        mlb,
    )

In [4]:
DATA_PATH = "/kaggle/input/datasets/omkarborikar/top-10000-popular-movies/Top_10000_Movies.csv"
(
    train_texts,
    val_texts,
    test_texts,
    y_train,
    y_val,
    y_test,
    mlb,
) = load_data_for_bert(
    DATA_PATH
)

print(
    "Train:",
    len(train_texts)
)

print(
    "Val:",
    len(val_texts)
)

print(
    "Test:",
    len(test_texts)
)

print(
    "Genres:",
    len(mlb.classes_)
)

Train: 5877
Val: 2938
Test: 981
Genres: 19


In [5]:
MODEL_NAME = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME
)

MAX_LENGTH = 256


def tokenize_function(
    texts
):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
train_encodings = tokenize_function(
    train_texts
)

val_encodings = tokenize_function(
    val_texts
)

test_encodings = tokenize_function(
    test_texts
)

train_dataset = Dataset.from_dict(
    {
        **train_encodings,
        "labels": y_train.astype(np.float32),
    }
)

val_dataset = Dataset.from_dict(
    {
        **val_encodings,
        "labels": y_val.astype(np.float32),
    }
)

test_dataset = Dataset.from_dict(
    {
        **test_encodings,
        "labels": y_test.astype(np.float32),
    }
)

In [7]:
def compute_metrics(
    eval_pred
):

    logits, labels = eval_pred

    probabilities = 1 / (
        1 + np.exp(-logits)
    )

    predictions = (
        probabilities > 0.20
  ).astype(int)

    micro_f1 = f1_score(
        labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    exact_match = accuracy_score(
        labels,
        predictions,
    )

    return {
        "micro_f1":
            micro_f1,
        "macro_f1":
            macro_f1,
        "exact_match":
            exact_match,
    }

In [8]:
#LEARNING RATE TESTING
'''
import time
import pandas as pd

learning_rates = [
    5e-5,
    4e-5,
    3e-5,
    2e-5,
]

results = []

for lr in learning_rates:

    print("\n" + "=" * 50)
    print(f"TRAINING WITH LR = {lr}")
    print("=" * 50)

    # Fresh model
    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
    )

    training_args = TrainingArguments(
        output_dir=f"./results_lr_{lr}",
        num_train_epochs=4,
        learning_rate=lr,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        metric_for_best_model="micro_f1",
        greater_is_better=True,
        fp16=True,
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    start_time = time.time()

    trainer.train()

    training_time = (
        time.time() - start_time
    )

    metrics = trainer.evaluate()

    results.append(
        {
            "learning_rate": lr,
            "micro_f1": metrics["eval_micro_f1"],
            "macro_f1": metrics["eval_macro_f1"],
            "exact_match": metrics["eval_exact_match"],
            "validation_loss": metrics["eval_loss"],
            "training_time_minutes":
                training_time / 60,
        }
    )

results_df = pd.DataFrame(results)

print("\nFINAL COMPARISON")
print(results_df)

results_df.to_csv(
    "learning_rate_results.csv",
    index=False,
)
'''

'\nimport time\nimport pandas as pd\n\nlearning_rates = [\n    5e-5,\n    4e-5,\n    3e-5,\n    2e-5,\n]\n\nresults = []\n\nfor lr in learning_rates:\n\n    print("\n" + "=" * 50)\n    print(f"TRAINING WITH LR = {lr}")\n    print("=" * 50)\n\n    # Fresh model\n    model = BertForSequenceClassification.from_pretrained(\n        MODEL_NAME,\n        num_labels=len(mlb.classes_),\n        problem_type="multi_label_classification",\n    )\n\n    training_args = TrainingArguments(\n        output_dir=f"./results_lr_{lr}",\n        num_train_epochs=4,\n        learning_rate=lr,\n        per_device_train_batch_size=16,\n        per_device_eval_batch_size=32,\n        weight_decay=0.01,\n        eval_strategy="epoch",\n        save_strategy="no",\n        load_best_model_at_end=False,\n        metric_for_best_model="micro_f1",\n        greater_is_better=True,\n        fp16=True,\n        logging_steps=50,\n        report_to="none",\n    )\n\n    trainer = Trainer(\n        model=model,\n     

In [9]:
#BATCH SIZE TESTING
'''
import time
import pandas as pd

batch_sizes = [
    8,
    16,
    32,
]

results = []

for batch_size in batch_sizes:

    print("\n" + "=" * 50)
    print(
        f"TRAINING WITH BATCH SIZE = {batch_size}"
    )
    print("=" * 50)

    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
    )

    training_args = TrainingArguments(
        output_dir=f"./batch_{batch_size}",

        num_train_epochs=4,

        learning_rate=5e-5,

        per_device_train_batch_size=batch_size,

        per_device_eval_batch_size=batch_size,

        weight_decay=0.01,

        eval_strategy="epoch",

        save_strategy="no",

        load_best_model_at_end=False,

        fp16=True,

        logging_steps=50,

        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
    )

    start_time = time.time()

    trainer.train()

    training_time = (
        time.time() - start_time
    )

    metrics = trainer.evaluate()

    results.append(
        {
            "batch_size":
                batch_size,
            "micro_f1":
                metrics["eval_micro_f1"],
            "macro_f1":
                metrics["eval_macro_f1"],
            "exact_match":
                metrics["eval_exact_match"],
            "validation_loss":
                metrics["eval_loss"],
            "training_time_minutes":
                training_time / 60,
        }
    )

results_df = pd.DataFrame(results)

print("\nFINAL COMPARISON")
print(results_df)

results_df.to_csv(
    "batch_size_results.csv",
    index=False,
)

results_df
'''

'\nimport time\nimport pandas as pd\n\nbatch_sizes = [\n    8,\n    16,\n    32,\n]\n\nresults = []\n\nfor batch_size in batch_sizes:\n\n    print("\n" + "=" * 50)\n    print(\n        f"TRAINING WITH BATCH SIZE = {batch_size}"\n    )\n    print("=" * 50)\n\n    model = BertForSequenceClassification.from_pretrained(\n        MODEL_NAME,\n        num_labels=len(mlb.classes_),\n        problem_type="multi_label_classification",\n    )\n\n    training_args = TrainingArguments(\n        output_dir=f"./batch_{batch_size}",\n\n        num_train_epochs=4,\n\n        learning_rate=5e-5,\n\n        per_device_train_batch_size=batch_size,\n\n        per_device_eval_batch_size=batch_size,\n\n        weight_decay=0.01,\n\n        eval_strategy="epoch",\n\n        save_strategy="no",\n\n        load_best_model_at_end=False,\n\n        fp16=True,\n\n        logging_steps=50,\n\n        report_to="none",\n    )\n\n    trainer = Trainer(\n        model=model,\n        args=training_args,\n        trai

In [10]:
model = (
    BertForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=len(
            mlb.classes_
        ),
        problem_type=
        "multi_label_classification",
    )
)

print(
    "Labels:",
    model.config.num_labels
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Using:",
    device
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Labels: 19
Using: cuda


In [11]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=8,

    learning_rate=5e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=32,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model=
    "micro_f1",

    greater_is_better=True,

    fp16=True,

    logging_steps=50,

    report_to="none",
)

In [12]:
import time

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

start_time = time.time()

trainer.train()

end_time = time.time()

training_time = end_time - start_time

print(
    f"Training time: {training_time:.2f} seconds"
)

print(
    f"Training time: {training_time / 60:.2f} minutes"
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Exact Match
1,0.519741,0.499990,0.612951,0.485390,0.056161
2,0.422739,0.463505,0.630059,0.578398,0.075221
3,0.328434,0.462775,0.638470,0.594979,0.083390
4,0.261017,0.466492,0.650831,0.606321,0.120490
5,0.198526,0.481127,0.656135,0.609470,0.120830
6,0.167718,0.486956,0.657433,0.612819,0.134786
7,0.131569,0.500153,0.656530,0.613030,0.131382
8,0.115214,0.502490,0.656442,0.612741,0.133764


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training time: 1645.40 seconds
Training time: 27.42 minutes


In [13]:
#TEST THRESHOLDS
predictions = trainer.predict(
    val_dataset
)

logits = (
    predictions.predictions
)

labels = (
    predictions.label_ids
)

probabilities = 1 / (
    1 + np.exp(-logits)
)

best_threshold = None
best_score = 0

for threshold in [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
]:

    preds = (
        probabilities >
        threshold
    ).astype(int)

    score = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0,
    )

    print(
        threshold,
        score,
    )

    if score > best_score:

        best_score = score
        best_threshold = threshold

print(
    "\nBest Threshold:",
    best_threshold
)

print(
    "Best Micro F1:",
    best_score
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


0.05 0.5848432055749129
0.1 0.6348516252970021
0.15 0.6506695818529653
0.2 0.6574707314245972
0.25 0.6590223295111648
0.3 0.6597590662255789
0.35 0.6590250499066264
0.4 0.6560720434379552
0.45 0.6532444565365256
0.5 0.6447957470621153

Best Threshold: 0.3
Best Micro F1: 0.6597590662255789


In [14]:
test_predictions = (
    trainer.predict(
        test_dataset
    )
)

logits = (
    test_predictions
    .predictions
)

labels = (
    test_predictions
    .label_ids
)

probabilities = 1 / (
    1 + np.exp(-logits)
)

preds = (
    probabilities >
    best_threshold
).astype(int)

micro_f1 = f1_score(
    labels,
    preds,
    average="micro",
)

macro_f1 = f1_score(
    labels,
    preds,
    average="macro",
)

exact_match = (
    accuracy_score(
        labels,
        preds
    )
)

print(
    "\nFINAL TEST RESULTS"
)

print(
    f"Micro-F1: {micro_f1:.4f}"
)

print(
    f"Macro-F1: {macro_f1:.4f}"
)

print(
    f"Exact Match: {exact_match:.4f}"
)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



FINAL TEST RESULTS
Micro-F1: 0.6560
Macro-F1: 0.6018
Exact Match: 0.1580


In [15]:
import os
import json
import pickle
import zipfile

# =====================================================
# CREATE SAVE DIRECTORY
# =====================================================

SAVE_DIR = "/kaggle/working/katebert_final"

os.makedirs(
    SAVE_DIR,
    exist_ok=True,
)

# =====================================================
# SAVE MODEL
# =====================================================

trainer.save_model(
    SAVE_DIR
)

# =====================================================
# SAVE TOKENIZER
# =====================================================

tokenizer.save_pretrained(
    SAVE_DIR
)

# =====================================================
# SAVE MULTILABEL BINARIZER
# =====================================================

with open(
    f"{SAVE_DIR}/mlb.pkl",
    "wb",
) as f:

    pickle.dump(
        mlb,
        f,
    )

# =====================================================
# SAVE RESULTS
# =====================================================

results = {
    "model": "bert-base-uncased",
    "best_threshold": float(
        best_threshold
    ),
    "test_micro_f1": float(
        micro_f1
    ),
    "test_macro_f1": float(
        macro_f1
    ),
    "test_exact_match": float(
        exact_match
    ),
    "num_labels": int(
        len(mlb.classes_)
    ),
    "genres": list(
        mlb.classes_
    ),
}

with open(
    f"{SAVE_DIR}/results.json",
    "w",
) as f:

    json.dump(
        results,
        f,
        indent=4,
    )

# =====================================================
# SAVE HUMAN-READABLE SUMMARY
# =====================================================

with open(
    f"{SAVE_DIR}/summary.txt",
    "w",
) as f:

    f.write(
        "KATEBERT FINAL RESULTS\n"
    )

    f.write(
        "======================\n\n"
    )

    f.write(
        f"Best Threshold: {best_threshold:.2f}\n"
    )

    f.write(
        f"Micro-F1: {micro_f1:.4f}\n"
    )

    f.write(
        f"Macro-F1: {macro_f1:.4f}\n"
    )

    f.write(
        f"Exact Match: {exact_match:.4f}\n"
    )

# =====================================================
# ZIP EVERYTHING
# =====================================================

zip_path = "/kaggle/working/katebert_final.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED,
) as zipf:

    for root, dirs, files in os.walk(
        SAVE_DIR
    ):

        for file in files:

            file_path = os.path.join(
                root,
                file,
            )

            archive_name = os.path.relpath(
                file_path,
                SAVE_DIR,
            )

            zipf.write(
                file_path,
                archive_name,
            )

# =====================================================
# PRINT WHAT WAS SAVED
# =====================================================

print(
    "\nSaved files:"
)

for file in sorted(
    os.listdir(SAVE_DIR)
):

    print(file)

print(
    f"\nZip file created:\n{zip_path}"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved files:
config.json
mlb.pkl
model.safetensors
results.json
summary.txt
tokenizer.json
tokenizer_config.json
training_args.bin

Zip file created:
/kaggle/working/katebert_final.zip
